# 🧭 OptiCompromise: Programación de Compromiso Multiobjetivo
### Asignatura: Introducción al Análisis de Decisiones (IAD) | Facultad de Minas – Universidad Nacional de Colombia
**Docente:** Patricia Jaramillo Álvarez, PhD | **Apoyo Académico:** Jaime Andrés Díaz Guerra, MSc  
**Autores:** Fabián Andrés Hurtado Arango (`1036450768`) & Oscar David Arango Garcia (`1011591946`)

---
## Presentación y Modo de Uso
Este cuaderno autónomo implementa la metodología de **Programación de Compromiso** sobre modelos lineales continuos y enteros mixtos (MIP), calculando:
- **Matriz de Pagos** con desempate lexicográfico, **Punto Ideal ($Z^*$)** y **Nadir aproximado ($Z_{\text{peor}}$)**.
- **Métrica $L_1$ (Manhattan):** Programación Lineal (HiGHS) con efecto compensatorio.
- **Métrica $L_2$ (Euclidiana):** Minimiza $\sum_i(w_iD_i)^2$ conforme a Clase 15 (pp. 9 y 19). En modelos enteros, la búsqueda tiene límites y puede devolver un punto factible sin certificado.
- **Métrica $L_\infty$ (Tchebysheff / Minimax):** Programación Lineal minimax con variable auxiliar $\lambda$ para reducir la mayor brecha ponderada.
- **Aproximación de Pareto:** Muestras filtradas entre las soluciones del barrido 2D/3D y perfil radial de grados de logro $Z'_i = 100\cdot(1 - D_i)$.

> Ejecuta todas las celdas en orden (**Entorno de ejecución → Ejecutar todas**). No requiere clonar repositorios externos.

In [ ]:
# 1. Importación de bibliotecas científicas libres (incluidas por defecto en Google Colab)
import sys
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print(f'✓ Entorno listo | Python {sys.version.split()[0]} | SciPy {scipy.__version__} (HiGHS + SLSQP)')

In [ ]:
# Motor generado por colab/sync_engine.py a partir de api/core.

"""
Esquemas Pydantic para definición de modelos multiobjetivo y respuestas de optimización.
"""
from typing import List, Dict, Optional, Literal, Tuple
from pydantic import BaseModel, Field

class VariableModel(BaseModel):
    name: str = Field(..., description="Identificador único de la variable (ej. x1, x2)")
    var_type: Literal["continuous", "integer", "binary"] = Field(
        default="continuous", description="Tipo de variable: continua, entera o binaria"
    )
    lower_bound: Optional[float] = Field(default=0.0, description="Cota inferior (None para sin límite)")
    upper_bound: Optional[float] = Field(default=None, description="Cota superior (None para sin límite)")

class ObjectiveModel(BaseModel):
    name: str = Field(..., description="Nombre del objetivo (ej. Utilidad, Costo, Emisiones)")
    sense: Literal["maximize", "minimize"] = Field(
        default="maximize", description="Sentido de optimización"
    )
    coefficients: List[float] = Field(
        default_factory=list, description="Coeficientes lineales correspondientes a cada variable"
    )
    expression: Optional[str] = Field(
        default=None, description="Expresión matemática opcional (para visualización o no lineales)"
    )

class ConstraintModel(BaseModel):
    name: str = Field(default="Restricción", description="Etiqueta descriptiva de la restricción")
    coefficients: List[float] = Field(
        default_factory=list, description="Coeficientes lineales correspondientes a cada variable"
    )
    sense: Literal["<=", ">=", "=="] = Field(default="<=", description="Sentido de la restricción")
    rhs: float = Field(..., description="Término independiente o lado derecho (Right Hand Side)")
    expression: Optional[str] = Field(default=None, description="Expresión opcional en texto")

class ModelDefinition(BaseModel):
    name: str = Field(default="Modelo Multiobjetivo", description="Nombre del problema de decisión")
    description: Optional[str] = Field(default="", description="Descripción del contexto del problema")
    reference: Optional[str] = None
    variables: List[VariableModel] = Field(..., description="Lista de variables de decisión")
    objectives: List[ObjectiveModel] = Field(..., description="Lista de funciones objetivo (al menos 2 o 3)")
    constraints: List[ConstraintModel] = Field(default_factory=list, description="Lista de restricciones")
    weights: Optional[List[float]] = Field(
        default=None, description="Pesos asignados a los objetivos (si se omite, se asignan pesos iguales)"
    )

    pareto_seed: int = Field(default=42, ge=0, le=2**32 - 1)

class PayoffMatrixData(BaseModel):
    headers: List[str]
    matrix: List[List[float]]  # Fila k: evaluación de todos los objetivos bajo la solución óptima del objetivo k
    ideal_point: List[float]   # Mejor valor encontrado para cada objetivo
    anti_ideal_point: List[float] # Clave histórica del nadir aproximado basado en pagos
    independent_solutions: List[Dict[str, float]]
    normalization_ranges: List[float] = Field(default_factory=list)
    normalization_sources: List[Literal["payoff", "feasible_range", "constant", "unit"]] = Field(default_factory=list)
    decision_ranges_2d: Optional[List[Tuple[Optional[float], Optional[float]]]] = None
    degenerate_objectives: List[int] = Field(
        default_factory=list,
        description="Índices (0-based) de objetivos cuyo rango Ideal-Nadir fue nulo antes del ajuste numérico"
    )
    status: str = "optimal"

class MetricSolution(BaseModel):
    metric: str  # "L1", "L2", "L_inf"
    variables: Dict[str, float]
    objectives: List[float]
    achievement_pct: List[float]  # 100 * (1 - D_i) [%]
    normalized_distances: List[float]  # D_i
    aggregate_distance: float
    status: str = "optimal"

class ParetoPoint(BaseModel):
    weights: List[float]
    objectives: List[float]
    variables: Dict[str, float]
    metric: str
    status: str = "optimal"

class SensitivityPoint(BaseModel):
    w1: float
    weights: List[float]
    deviations_pct: List[float]
    objectives: List[float]
    status: str

class WeightSensitivityData(BaseModel):
    objective_index: int
    objective_name: str
    objective_names: List[str]
    l1: List[SensitivityPoint]
    l2: List[SensitivityPoint]
    l_inf: List[SensitivityPoint]

class SolveResponse(BaseModel):
    status: str
    message: str
    payoff: PayoffMatrixData
    solutions: Dict[str, MetricSolution]  # "L1", "L2", "L_inf"
    weights: List[float]
    pareto_points: Optional[List[ParetoPoint]] = None
    weight_sensitivity: Optional[WeightSensitivityData] = None

"""
Cálculo de la Matriz de Pagos, Punto Ideal y Nadir aproximado.
"""
import numpy as np
from scipy.optimize import linprog
from typing import Tuple, List, Dict, Optional

def scale_linear_objective(coefficients: np.ndarray) -> np.ndarray:
    """Usa una escala positiva para que el solver no trate todo el costo como cero."""
    scale = float(np.max(np.abs(coefficients), initial=0.0))
    return coefficients / scale if scale else coefficients


def build_linear_constraints(model: ModelDefinition) -> Tuple[
    Optional[np.ndarray], Optional[np.ndarray],
    Optional[np.ndarray], Optional[np.ndarray],
    List[Tuple[Optional[float], Optional[float]]],
    Optional[List[int]]
]:
    """
    Construye las matrices y vectores estándar para scipy.optimize.linprog:
    A_ub @ x <= b_ub
    A_eq @ x == b_eq
    bounds y vector de integrality (para MIP).
    """
    n_vars = len(model.variables)
    
    A_ub_list, b_ub_list = [], []
    A_eq_list, b_eq_list = [], []
    
    for c in model.constraints:
        coeffs = list(c.coefficients)
        if len(coeffs) < n_vars:
            coeffs += [0.0] * (n_vars - len(coeffs))
        coeffs = np.array(coeffs[:n_vars], dtype=float)
        
        rhs = float(c.rhs)
        # Equivalent rows should have the same numerical representation.
        scale = np.max(np.abs(coeffs), initial=0.0)
        if scale and np.isfinite(rhs / scale):
            coeffs = coeffs / scale
            rhs /= scale

        if c.sense == "<=":
            A_ub_list.append(coeffs)
            b_ub_list.append(rhs)
        elif c.sense == ">=":
            A_ub_list.append(-coeffs)
            b_ub_list.append(-rhs)
        elif c.sense == "==":
            A_eq_list.append(coeffs)
            b_eq_list.append(rhs)
            
    A_ub = np.array(A_ub_list) if A_ub_list else None
    b_ub = np.array(b_ub_list) if b_ub_list else None
    A_eq = np.array(A_eq_list) if A_eq_list else None
    b_eq = np.array(b_eq_list) if b_eq_list else None
    
    # Cotas y tipos de variables
    bounds = []
    integrality = []
    has_integer = False
    
    for v in model.variables:
        lb = v.lower_bound
        ub = v.upper_bound
        if v.var_type == "binary":
            lb = 0.0 if lb is None else max(0.0, lb)
            ub = 1.0 if ub is None else min(1.0, ub)
            integrality.append(1)
            has_integer = True
        elif v.var_type == "integer":
            integrality.append(1)
            has_integer = True
        else:
            integrality.append(0)
        bounds.append((lb, ub))
        
    integrality_vector = integrality if has_integer else None
    return A_ub, b_ub, A_eq, b_eq, bounds, integrality_vector

def compute_payoff_matrix(model: ModelDefinition) -> PayoffMatrixData:
    """
    Resuelve cada objetivo mono-objetivo de forma independiente y genera la matriz de pagos.
    """
    n_vars = len(model.variables)
    n_objs = len(model.objectives)
    A_ub, b_ub, A_eq, b_eq, bounds, integrality = build_linear_constraints(model)
    
    payoff_matrix = np.zeros((n_objs, n_objs), dtype=float)
    independent_solutions: List[Dict[str, float]] = []
    
    headers = [obj.name for obj in model.objectives]
    var_names = [v.name for v in model.variables]
    
    # 1. Optimización independiente de cada objetivo (con desempate lexicográfico para evitar soluciones dominadas)
    obj_dirs: List[np.ndarray] = []
    for obj in model.objectives:
        coeffs = list(obj.coefficients)
        if len(coeffs) < n_vars:
            coeffs += [0.0] * (n_vars - len(coeffs))
        arr = np.array(coeffs[:n_vars], dtype=float)
        obj_dirs.append(scale_linear_objective(-arr if obj.sense == "maximize" else arr))

    for k, obj in enumerate(model.objectives):
        c_solver = obj_dirs[k]

        res = linprog(
            c_solver,
            A_ub=A_ub,
            b_ub=b_ub,
            A_eq=A_eq,
            b_eq=b_eq,
            bounds=bounds,
            integrality=integrality,
            method="highs"
        )

        if not res.success:
            return PayoffMatrixData(
                headers=headers,
                matrix=[],
                ideal_point=[],
                anti_ideal_point=[],
                independent_solutions=[],
                degenerate_objectives=[],
                status=f"error: {res.message} en objetivo {obj.name}"
            )

        x_opt = np.array(res.x, dtype=float)

        # Desempate lexicográfico secundario: entre todos los óptimos alternativos de Z_k,
        # elegir el que mejor desempeña en los demás objetivos para no fabricar un Nadir dominado.
        if n_objs >= 2:
            z_k_opt = float(np.dot(c_solver, x_opt))
            secondary = np.zeros(n_vars, dtype=float)
            for j in range(n_objs):
                if j == k:
                    continue
                norm_j = float(np.linalg.norm(obj_dirs[j]))
                if norm_j > 1e-12:
                    secondary += obj_dirs[j] / norm_j

            if np.linalg.norm(secondary) > 1e-12:
                A_eq_lex = (
                    np.vstack([A_eq, c_solver[np.newaxis, :]])
                    if A_eq is not None and len(A_eq) > 0
                    else c_solver[np.newaxis, :]
                )
                b_eq_lex = (
                    np.concatenate([b_eq, [z_k_opt]])
                    if b_eq is not None and len(b_eq) > 0
                    else np.array([z_k_opt], dtype=float)
                )
                res_lex = linprog(
                    secondary,
                    A_ub=A_ub,
                    b_ub=b_ub,
                    A_eq=A_eq_lex,
                    b_eq=b_eq_lex,
                    bounds=bounds,
                    integrality=integrality,
                    method="highs"
                )
                if res_lex.success and res_lex.x is not None:
                    z_k_lex = float(np.dot(c_solver, res_lex.x))
                    if z_k_lex <= z_k_opt + 1e-9 * max(1.0, abs(z_k_opt)):
                        x_opt = np.array(res_lex.x, dtype=float)

        # Limpiar tolerancias de integralidad y ruido de cotas
        if integrality is not None:
            for j, flag in enumerate(integrality):
                if flag == 1 and abs(x_opt[j] - round(float(x_opt[j]))) < 1e-5:
                    x_opt[j] = float(round(float(x_opt[j])))
        for j, (lb, ub) in enumerate(bounds):
            if lb is not None and abs(x_opt[j] - lb) < 1e-9:
                x_opt[j] = lb
            if ub is not None and abs(x_opt[j] - ub) < 1e-9:
                x_opt[j] = ub

        independent_solutions.append({var_names[j]: float(x_opt[j]) for j in range(n_vars)})

        # Evaluar todos los objetivos bajo esta solución óptima x^(k)*
        for i, eval_obj in enumerate(model.objectives):
            eval_coeffs = list(eval_obj.coefficients)
            if len(eval_coeffs) < n_vars:
                eval_coeffs += [0.0] * (n_vars - len(eval_coeffs))
            eval_coeffs = np.array(eval_coeffs[:n_vars], dtype=float)
            val = float(np.dot(eval_coeffs, x_opt))
            payoff_matrix[k, i] = val

    # 2. Determinar punto ideal, referencia de pagos y escalas
    ideal_point = []
    anti_ideal_point = []
    degenerate_objectives: List[int] = []
    normalization_ranges = []
    normalization_sources = []

    payoff_vectors = np.abs(np.array([list(row.values()) for row in independent_solutions]))
    for i, obj in enumerate(model.objectives):
        col_vals = payoff_matrix[:, i]
        if obj.sense == "maximize":
            z_ideal = float(np.max(col_vals))
            z_anti = float(np.min(col_vals))
        else:
            z_ideal = float(np.min(col_vals))
            z_anti = float(np.max(col_vals))

        scale = abs(z_ideal - z_anti)
        source = "payoff"
        # Comparar con el redondeo de la evaluación, en unidades del criterio.
        # Una tolerancia absoluta anularía objetivos pequeños pero válidos.
        coefficients = np.abs(np.asarray(obj.coefficients, dtype=float))
        magnitude = float(np.max(payoff_vectors[:, :len(coefficients)] @ coefficients))
        roundoff = 16 * np.finfo(float).eps * magnitude
        if scale <= roundoff:
            degenerate_objectives.append(i)
            # La referencia de pagos se conserva. Solo se sustituye el denominador.
            worst = linprog(
                -obj_dirs[i], A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                bounds=bounds, integrality=integrality, method="highs",
            )
            if worst.success:
                original = np.asarray(obj.coefficients, dtype=float)
                z_worst = float(original @ worst.x[:len(original)])
                scale = max(0.0, z_ideal - z_worst if obj.sense == "maximize" else z_worst - z_ideal)
                worst_magnitude = float(np.abs(original) @ np.abs(worst.x[:len(original)]))
                if scale > max(roundoff, 16 * np.finfo(float).eps * worst_magnitude):
                    source = "feasible_range"
                else:
                    scale, source = 1.0, "constant"
            else:
                # Sin peor extremo verificable, una unidad del objetivo sirve de escala explícita.
                scale, source = 1.0, "unit"
        normalization_ranges.append(scale)
        normalization_sources.append(source)

        ideal_point.append(z_ideal)
        anti_ideal_point.append(z_anti)

    decision_ranges = None
    if n_vars == 2:
        decision_ranges = []
        for axis in np.eye(2):
            endpoints = []
            for direction in (1, -1):
                projection = linprog(
                    direction * axis, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                    bounds=bounds, method="highs",
                )
                endpoints.append(float(direction * projection.fun) if projection.success else None)
            decision_ranges.append(tuple(endpoints))

    return PayoffMatrixData(
        headers=headers,
        matrix=payoff_matrix.tolist(),
        ideal_point=ideal_point,
        anti_ideal_point=anti_ideal_point,
        independent_solutions=independent_solutions,
        degenerate_objectives=degenerate_objectives,
        normalization_ranges=normalization_ranges,
        normalization_sources=normalization_sources,
        decision_ranges_2d=decision_ranges,
        status="optimal"
    )

"""
Solvers para Programación de Compromiso Multiobjetivo:
Métricas L1 (Lineal), L_inf (Minimax Lineal) y L2 (Cuadrática Convexa / MIQP Branch-and-Bound).
"""
import heapq
import time
import numpy as np
from scipy.linalg import qr
from scipy.optimize import Bounds, LinearConstraint, linprog, milp, minimize
from typing import List, Dict, Tuple, Optional

def compute_distance_coefficients(
    model: ModelDefinition, payoff: PayoffMatrixData
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Calcula los vectores alfa_i y escalares beta_i tales que:
    D_i(x) = alfa_i^T @ x + beta_i
    donde D_i(x) es la distancia normalizada al ideal (D_i = 0 en el ideal, D_i = 1 en el peor).
    """
    n_vars = len(model.variables)
    n_objs = len(model.objectives)
    
    alpha = np.zeros((n_objs, n_vars), dtype=float)
    beta = np.zeros(n_objs, dtype=float)
    
    for i, obj in enumerate(model.objectives):
        if payoff.normalization_sources and payoff.normalization_sources[i] == "constant":
            continue
        coeffs = list(obj.coefficients)
        if len(coeffs) < n_vars:
            coeffs += [0.0] * (n_vars - len(coeffs))
        c_i = np.array(coeffs[:n_vars], dtype=float)
        
        z_id = payoff.ideal_point[i]
        z_pe = payoff.anti_ideal_point[i]
        range_i = z_id - z_pe if obj.sense == "maximize" else z_pe - z_id
        
        if payoff.normalization_ranges:
            range_i = payoff.normalization_ranges[i]
        range_i = range_i or 1.0
            
        if obj.sense == "maximize":
            # D_i = (z_id - c_i^T x) / (z_id - z_pe)
            alpha[i] = - c_i / range_i
            beta[i] = z_id / range_i
        else:
            # D_i = (c_i^T x - z_id) / (z_pe - z_id)
            alpha[i] = c_i / range_i
            beta[i] = - z_id / range_i
            
    return alpha, beta

def format_solution(
    metric_name: str,
    x_opt: np.ndarray,
    model: ModelDefinition,
    payoff: PayoffMatrixData,
    alpha: np.ndarray,
    beta: np.ndarray,
    weights: np.ndarray,
    status: str = "optimal"
) -> MetricSolution:
    """
    Construye el objeto MetricSolution evaluando variables, objetivos, distancias y grado de logro %.
    """
    var_names = [v.name for v in model.variables]
    variables_dict = {var_names[j]: float(x_opt[j]) for j in range(len(var_names))}
    
    objectives_vals = []
    norm_distances = []
    achievement_pcts = []
    
    for i, obj in enumerate(model.objectives):
        coeffs = list(obj.coefficients)
        if len(coeffs) < len(x_opt):
            coeffs += [0.0] * (len(x_opt) - len(coeffs))
        c_i = np.array(coeffs[:len(x_opt)], dtype=float)
        
        z_val = float(np.dot(c_i, x_opt))
        d_val = float(np.dot(alpha[i], x_opt) + beta[i])
        # Asegurar no negatividad de distancias por estabilidad numérica
        d_val = max(0.0, d_val)
        # Grado de logro: 100 * (1 - D_i) [%]
        ach_pct = float(100.0 * (1.0 - d_val))
        
        objectives_vals.append(z_val)
        norm_distances.append(d_val)
        achievement_pcts.append(ach_pct)
        
    d_array = np.array(norm_distances)
    if metric_name == "L1":
        agg_dist = float(np.sum(weights * d_array))
    elif metric_name == "L2":
        agg_dist = float(np.linalg.norm(weights * d_array))
    else:  # L_inf
        agg_dist = float(np.max(weights * d_array))
        
    return MetricSolution(
        metric=metric_name,
        variables=variables_dict,
        objectives=objectives_vals,
        achievement_pct=achievement_pcts,
        normalized_distances=norm_distances,
        aggregate_distance=agg_dist,
        status=status
    )

def _clean_solution_vector(
    x: np.ndarray,
    bounds: List[Tuple[Optional[float], Optional[float]]],
    integrality: Optional[List[int]] = None,
) -> np.ndarray:
    """Limpia tolerancias numéricas en cotas y variables enteras/binarias."""
    cleaned = np.array(x, dtype=float)
    if integrality is not None:
        for j, flag in enumerate(integrality):
            if flag == 1 and abs(cleaned[j] - round(float(cleaned[j]))) < 1e-5:
                cleaned[j] = float(round(float(cleaned[j])))
    for j, (lb, ub) in enumerate(bounds):
        if lb is not None and abs(cleaned[j] - lb) < 1e-9:
            cleaned[j] = lb
        if ub is not None and abs(cleaned[j] - ub) < 1e-9:
            cleaned[j] = ub
    return cleaned


def _continuous_column_scale(model, payoff, bounds, integrality, minimum=1e4):
    # Los óptimos individuales también revelan cotas efectivas impuestas por filas.
    # Las coordenadas enteras conservan su retícula original.
    scales = np.ones(len(bounds))
    for j, (variable, (lo, hi)) in enumerate(zip(model.variables, bounds)):
        if integrality is not None and integrality[j]:
            continue
        witness = max((abs(row[variable.name]) for row in payoff.independent_solutions), default=0.0)
        scale = max(witness, abs(lo or 0.0), abs(hi or 0.0))
        if scale >= minimum:
            scales[j] = scale
    return scales


def _scale_bounds(bounds, column_scale):
    return [
        tuple(None if bound is None else bound / scale for bound in pair)
        for pair, scale in zip(bounds, column_scale)
    ]


def solve_l1(
    model: ModelDefinition,
    payoff: PayoffMatrixData,
    alpha: np.ndarray,
    beta: np.ndarray,
    weights: np.ndarray
) -> MetricSolution:
    """
    Resuelve la métrica L1 (Manhattan) como Programación Lineal:
    Min sum(w_i * D_i(x)) = Min (sum(w_i * alpha_i))^T @ x
    Con pesos nulos, desempata dentro de la cara óptima sin alterar la métrica.
    """
    A_ub, b_ub, A_eq, b_eq, bounds, integrality = build_linear_constraints(model)

    c_l1 = scale_linear_objective(weights @ alpha)

    res = linprog(
        c_l1,
        A_ub=A_ub,
        b_ub=b_ub,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        integrality=integrality,
        method="highs"
    )

    if not res.success:
        return MetricSolution(
            metric="L1",
            variables={},
            objectives=[],
            achievement_pct=[],
            normalized_distances=[],
            aggregate_distance=0.0,
            status=f"error: {res.message}"
        )

    x = res.x
    if np.any(weights == 0):
        if np.any(c_l1):
            A_eq = np.vstack([A_eq, c_l1]) if A_eq is not None else c_l1[None, :]
            b_eq = np.append(b_eq, c_l1 @ x) if b_eq is not None else [c_l1 @ x]
        tie = linprog(scale_linear_objective(alpha.sum(axis=0)), A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                      bounds=bounds, integrality=integrality, method="highs")
        if tie.success:
            x = tie.x
    x_clean = _clean_solution_vector(x, bounds, integrality)
    return format_solution("L1", x_clean, model, payoff, alpha, beta, weights, status="optimal")

def solve_linf(
    model: ModelDefinition,
    payoff: PayoffMatrixData,
    alpha: np.ndarray,
    beta: np.ndarray,
    weights: np.ndarray
) -> MetricSolution:
    """
    Resuelve la métrica L_inf (Tchebysheff / Minimax) mediante Programación Lineal con variable auxiliar lambda:
    Min lambda, con desempate en la cara óptima para mejorar la eficiencia.
    s.a. w_i * (alpha_i^T @ x + beta_i) <= lambda  <=>  w_i * alpha_i^T @ x - lambda <= - w_i * beta_i
         restricciones originales sobre x
    """
    n_vars = len(model.variables)
    n_objs = len(model.objectives)
    A_ub_orig, b_ub_orig, A_eq_orig, b_eq_orig, bounds_orig, integrality_orig = build_linear_constraints(model)

    column_scale = _continuous_column_scale(model, payoff, bounds_orig, integrality_orig)
    scaled_bounds = _scale_bounds(bounds_orig, column_scale)

    c_inf = np.zeros(n_vars + 1, dtype=float)
    c_inf[-1] = 1.0

    # Restricciones de desigualdad ampliadas
    A_ub_list = []
    b_ub_list = []

    if A_ub_orig is not None and len(A_ub_orig) > 0:
        A_ub_padded = np.hstack([A_ub_orig * column_scale, np.zeros((len(A_ub_orig), 1))])
        A_ub_list.append(A_ub_padded)
        b_ub_list.append(b_ub_orig)

    # Restricciones auxiliares de minimax: w_i * alpha_i^T @ x - lambda <= - w_i * beta_i
    A_minimax = np.hstack([weights[:, np.newaxis] * alpha * column_scale, -np.ones((n_objs, 1))])
    b_minimax = - weights * beta

    A_ub_list.append(A_minimax)
    b_ub_list.append(b_minimax)

    A_ub_total = np.vstack(A_ub_list)
    b_ub_total = np.concatenate(b_ub_list)

    # Restricciones de igualdad ampliadas
    A_eq_total = None
    b_eq_total = None
    if A_eq_orig is not None and len(A_eq_orig) > 0:
        A_eq_total = np.hstack([A_eq_orig * column_scale, np.zeros((len(A_eq_orig), 1))])
        b_eq_total = b_eq_orig

    # Cotas ampliadas: cotas de x y lambda >= 0
    bounds_total = scaled_bounds + [(0.0, None)]

    # Integrality ampliada (lambda es continua)
    integrality_total = None
    if integrality_orig is not None:
        integrality_total = list(integrality_orig) + [0]

    res = linprog(
        c_inf,
        A_ub=A_ub_total,
        b_ub=b_ub_total,
        A_eq=A_eq_total,
        b_eq=b_eq_total,
        bounds=bounds_total,
        integrality=integrality_total,
        method="highs"
    )

    if not res.success:
        return MetricSolution(
            metric="L_inf",
            variables={},
            objectives=[],
            achievement_pct=[],
            normalized_distances=[],
            aggregate_distance=0.0,
            status=f"error: {res.message}"
        )

    tie = linprog(
        np.append(scale_linear_objective(alpha.sum(axis=0) * column_scale), 0.0),
        A_ub=A_ub_total,
        b_ub=b_ub_total,
        A_eq=A_eq_total,
        b_eq=b_eq_total,
        bounds=scaled_bounds + [(res.x[-1], res.x[-1])],
        integrality=integrality_total,
        method="highs",
    )
    x = tie.x if tie.success else res.x
    x_opt = _clean_solution_vector(x[:n_vars] * column_scale, bounds_orig, integrality_orig)
    if not _is_feasible(x_opt, A_ub_orig, b_ub_orig, A_eq_orig, b_eq_orig, bounds_orig, integrality_orig):
        return MetricSolution(metric="L_inf", variables={}, objectives=[], achievement_pct=[],
                              normalized_distances=[], aggregate_distance=0.0,
                              status="error: la solución minimax no cumple el modelo original")
    distance = np.max(weights * (alpha @ x_opt + beta))
    status = "optimal" if abs(distance - res.x[-1]) <= 1e-7 else "feasible_suboptimal"
    return format_solution("L_inf", x_opt, model, payoff, alpha, beta, weights, status=status)

def _is_feasible(
    x: np.ndarray,
    A_ub: Optional[np.ndarray],
    b_ub: Optional[np.ndarray],
    A_eq: Optional[np.ndarray],
    b_eq: Optional[np.ndarray],
    bounds: List[Tuple[Optional[float], Optional[float]]],
    integrality: Optional[List[int]] = None,
    tol: float = 1e-7,
) -> bool:
    """Verifica estrictamente factibilidad (cotas, desigualdades, igualdades e integralidad)."""
    if x is None or len(x) != len(bounds) or not np.all(np.isfinite(x)):
        return False
    for j, (lb, ub) in enumerate(bounds):
        if lb is not None and x[j] < lb - tol:
            return False
        if ub is not None and x[j] > ub + tol:
            return False
    if A_ub is not None and b_ub is not None and len(b_ub) > 0:
        scale_ub = np.maximum(np.linalg.norm(A_ub, axis=1), np.maximum(np.abs(b_ub), 1.0))
        if np.any((A_ub @ x - b_ub) / scale_ub > tol):
            return False
    if A_eq is not None and b_eq is not None and len(b_eq) > 0:
        scale_eq = np.maximum(np.linalg.norm(A_eq, axis=1), np.maximum(np.abs(b_eq), 1.0))
        if np.any(np.abs(A_eq @ x - b_eq) / scale_eq > tol):
            return False
    if integrality is not None:
        for j, is_int in enumerate(integrality):
            if is_int == 1 and abs(x[j] - round(float(x[j]))) > 1e-7:
                return False
    return True


def _solve_continuous_qp(
    alpha: np.ndarray,
    beta: np.ndarray,
    weights: np.ndarray,
    A_ub: Optional[np.ndarray],
    b_ub: Optional[np.ndarray],
    A_eq: Optional[np.ndarray],
    b_eq: Optional[np.ndarray],
    bounds: List[Tuple[Optional[float], Optional[float]]],
    x0_candidates: List[np.ndarray],
) -> Tuple[Optional[np.ndarray], float, bool]:
    """
    Resuelve la relajación cuadrática continua Min sum((w_i * (alpha_i^T x + beta_i))^2)
    con restricciones escaladas por fila para estabilidad numérica de SLSQP.
    """
    n_vars = len(bounds)

    # Verificar que las cotas no sean vacías (lb > ub)
    for lb, ub in bounds:
        if lb is not None and ub is not None and lb > ub + 1e-9:
            return None, float("inf"), True

    weighted_alpha = weights[:, None] * alpha
    weighted_beta = weights * beta

    def objective(x: np.ndarray) -> float:
        d = weighted_alpha @ x + weighted_beta
        return float(d @ d)

    def jacobian(x: np.ndarray) -> np.ndarray:
        d = weighted_alpha @ x + weighted_beta
        return 2.0 * (weighted_alpha.T @ d)

    # Normalización de restricciones por norma de fila para evitar mal condicionamiento (ej. 2.35M vs 50)
    constraints = []
    if A_ub is not None and b_ub is not None:
        for i in range(len(b_ub)):
            row = A_ub[i]
            rhs = float(b_ub[i])
            scale = float(max(np.linalg.norm(row), abs(rhs), 1.0))
            r_s = row / scale
            b_s = rhs / scale
            constraints.append({
                "type": "ineq",
                "fun": lambda x, r=r_s, b=b_s: b - float(np.dot(r, x)),
                "jac": lambda x, r=r_s: -r,
            })

    if A_eq is not None and b_eq is not None:
        # SLSQP requiere igualdades independientes. Se conserva el sistema
        # completo para la comprobación de factibilidad y para HiGHS.
        scaled_eq = A_eq / np.maximum(np.linalg.norm(A_eq, axis=1, keepdims=True), 1e-300)
        rank = np.linalg.matrix_rank(scaled_eq)
        _, _, pivots = qr(scaled_eq.T, mode="economic", pivoting=True)
        for i in pivots[:rank]:
            row = A_eq[i]
            rhs = float(b_eq[i])
            scale = float(max(np.linalg.norm(row), abs(rhs), 1.0))
            r_s = row / scale
            b_s = rhs / scale
            constraints.append({
                "type": "eq",
                "fun": lambda x, r=r_s, b=b_s: float(np.dot(r, x)) - b,
                "jac": lambda x, r=r_s: r,
            })

    best_x: Optional[np.ndarray] = None
    best_val = float("inf")

    # Proyectar candidatos iniciales dentro de las cotas del nodo
    starts: List[np.ndarray] = []
    for cand in x0_candidates:
        if cand is not None and len(cand) == n_vars and np.all(np.isfinite(cand)):
            proj = np.array(cand, dtype=float)
            for j, (lb, ub) in enumerate(bounds):
                if lb is not None:
                    proj[j] = max(proj[j], lb)
                if ub is not None:
                    proj[j] = min(proj[j], ub)
            starts.append(proj)
    if not starts:
        starts.append(np.zeros(n_vars, dtype=float))

    # Sembrar best_x con el mejor candidato factible inicial (garantiza que L2 nunca empeore a L1/L_inf)
    for x_init in starts:
        if _is_feasible(x_init, A_ub, b_ub, A_eq, b_eq, bounds, integrality=None, tol=1e-7):
            val = objective(x_init)
            if val < best_val:
                best_val = val
                best_x = np.array(x_init, dtype=float)

    for x_init in starts[:2]:
        res = minimize(
            objective,
            x_init,
            jac=jacobian,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 400, "ftol": 1e-11},
        )
        if res.x is not None and _is_feasible(res.x, A_ub, b_ub, A_eq, b_eq, bounds, integrality=None, tol=1e-7):
            val = objective(res.x)
            if val < best_val:
                best_val = val
                best_x = np.array(res.x, dtype=float)

    # Si SLSQP no encontró punto factible, intentar encontrar un punto factible con HiGHS y re-optimizar
    if best_x is None:
        feas = linprog(
            np.zeros(n_vars, dtype=float),
            A_ub=A_ub,
            b_ub=b_ub,
            A_eq=A_eq,
            b_eq=b_eq,
            bounds=bounds,
            method="highs",
        )
        if not feas.success or feas.x is None:
            return None, float("inf"), feas.status == 2
        best_x = np.array(feas.x, dtype=float)
        best_val = objective(best_x)
        res = minimize(
            objective,
            best_x,
            jac=jacobian,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 400, "ftol": 1e-10},
        )
        if res.x is not None and _is_feasible(res.x, A_ub, b_ub, A_eq, b_eq, bounds, integrality=None, tol=1e-7):
            val = objective(res.x)
            if val < best_val:
                best_val = val
                best_x = np.array(res.x, dtype=float)

    # Para una cuadrática convexa, la tangente es un minorante global.
    # Su mínimo sobre el poliedro certifica una cota inferior para la poda.
    gradient = jacobian(best_x)
    scale = float(np.max(np.abs(gradient)))
    if scale == 0.0:
        return best_x, best_val, True
    certificate = linprog(
        gradient / scale, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
        bounds=bounds, method="highs",
    )
    if not certificate.success:
        return best_x, 0.0, False
    gap = max(0.0, float(gradient @ (best_x - certificate.x)))
    return best_x, max(0.0, best_val - gap), gap <= 1e-7


def _refine_integer_qp(alpha, beta, linear_rows, bounds, integrality, seeds,
                       incumbent, max_rounds, deadline):
    """Acota la suma de cuadrados con tangentes y resuelve los cortes como MILP."""
    A_ub, b_ub, A_eq, b_eq = linear_rows
    n_obj, n_var = alpha.shape
    rows, lower, upper = [], [], []
    if A_ub is not None:
        rows.extend(np.pad(A_ub, ((0, 0), (0, n_obj))))
        lower.extend([-np.inf] * len(b_ub))
        upper.extend(b_ub)
    if A_eq is not None:
        rows.extend(np.pad(A_eq, ((0, 0), (0, n_obj))))
        lower.extend(b_eq)
        upper.extend(b_eq)

    def add_support(x):
        d = alpha @ x + beta
        # t_i >= d_i^2 + 2*d_i*((alpha*x + beta)_i - d_i)
        rows.extend(np.column_stack((2 * d[:, None] * alpha, -np.eye(n_obj))))
        lower.extend([-np.inf] * n_obj)
        upper.extend(d * (d - 2 * beta))

    for seed in seeds:
        add_support(seed)
    lb = [lo if lo is not None else -np.inf for lo, _ in bounds]
    ub = [hi if hi is not None else np.inf for _, hi in bounds]
    extended_bounds = Bounds(lb + [0.] * n_obj, ub + [np.inf] * n_obj)
    extended_integrality = np.r_[integrality, np.zeros(n_obj, dtype=int)]
    cost = np.r_[np.zeros(n_var), np.ones(n_obj)]
    best_x, best_val = incumbent

    for _ in range(max_rounds):
        remaining = deadline - time.monotonic()
        if remaining <= 0:
            break
        result = milp(cost, integrality=extended_integrality, bounds=extended_bounds,
                      constraints=LinearConstraint(rows, lower, upper),
                      options={"time_limit": remaining, "mip_rel_gap": 0.0})
        if result.x is None:
            break
        x = _clean_solution_vector(result.x[:n_var], bounds, integrality)
        if not _is_feasible(x, A_ub, b_ub, A_eq, b_eq, bounds, integrality):
            break
        d = alpha @ x + beta
        value = float(d @ d)
        if value < best_val:
            best_x, best_val = x, value
        bound = getattr(result, "mip_dual_bound", None)
        if bound is not None and np.isfinite(bound) and best_val - bound <= 1e-7:
            return best_x, best_val, True
        add_support(x)

    return best_x, best_val, False


def solve_l2(
    model: ModelDefinition,
    payoff: PayoffMatrixData,
    alpha: np.ndarray,
    beta: np.ndarray,
    weights: np.ndarray,
    x0: Optional[np.ndarray] = None,
    incumbents: Optional[List[np.ndarray]] = None,
    max_nodes: int = 800,
    max_seconds: float = 1.0,
) -> MetricSolution:
    """
    Resuelve la métrica L2 (Euclidiana) como Programación Cuadrática Convexa (QP):
    Min sum((w_i * (alpha_i^T @ x + beta_i))^2)
    s.a. restricciones originales.
    - Para LP continuo: SLSQP con restricciones normalizadas y validación estricta de factibilidad.
    - Para MIP: cortes externos y ramificación y poda con presupuesto acotado.
    """
    n_vars = len(model.variables)
    var_names = [v.name for v in model.variables]
    A_ub_orig, b_ub, A_eq_orig, b_eq, bounds_orig, integrality = build_linear_constraints(model)
    # Llevar las variables continuas a una escala manejable evita que
    # SLSQP se detenga prematuramente con gradientes pequeños en unidades físicas.
    column_scale = _continuous_column_scale(model, payoff, bounds_orig, integrality, minimum=1.0)
    bounds = _scale_bounds(bounds_orig, column_scale)
    A_ub = None if A_ub_orig is None else A_ub_orig * column_scale
    A_eq = None if A_eq_orig is None else A_eq_orig * column_scale
    scaled_alpha = alpha * column_scale

    def objective(x: np.ndarray) -> float:
        d = scaled_alpha @ x + beta
        return float(np.sum((weights * d) ** 2))

    def report(x: np.ndarray, certified: bool) -> MetricSolution:
        x = _clean_solution_vector(x * column_scale, bounds_orig, integrality)
        if not _is_feasible(x, A_ub_orig, b_ub, A_eq_orig, b_eq, bounds_orig, integrality):
            return MetricSolution(metric="L2", variables={}, objectives=[], normalized_distances=[],
                                  achievement_pct=[], aggregate_distance=0.0,
                                  status="error: la solución L2 no cumple el modelo original")
        return format_solution("L2", x, model, payoff, alpha, beta, weights,
                               status="optimal" if certified else "feasible_suboptimal")

    # Construir candidatos iniciales factibles
    x0_candidates: List[np.ndarray] = []
    if x0 is not None and len(x0) == n_vars:
        x0_candidates.append(np.array(x0, dtype=float) / column_scale)
    if incumbents:
        for inc in incumbents:
            if inc is not None and len(inc) == n_vars:
                x0_candidates.append(np.array(inc, dtype=float) / column_scale)
    if payoff.independent_solutions:
        ind_vecs = [np.array([sol[name] for name in var_names], dtype=float) for sol in payoff.independent_solutions]
        x0_candidates.append(np.mean(ind_vecs, axis=0) / column_scale)

    # --- CASO 1: CONTINUO PURO ---
    if integrality is None:
        best_x, _, certified = _solve_continuous_qp(scaled_alpha, beta, weights, A_ub, b_ub, A_eq, b_eq, bounds, x0_candidates)
        if best_x is None:
            return MetricSolution(
                metric="L2",
                variables={},
                objectives=[],
                achievement_pct=[],
                normalized_distances=[],
                aggregate_distance=0.0,
                status="error: no se encontró solución factible L2",
            )
        return report(best_x, certified)

    # --- CASO 2: MIP (ENTERO / BINARIO) — RAMIFICACIÓN Y PODA CON PRESUPUESTO ---
    int_indices = [j for j, flag in enumerate(integrality) if flag == 1]

    best_int_x: Optional[np.ndarray] = None
    best_int_val = float("inf")

    # Sembrar cota superior (incumbente) con soluciones enteras ya conocidas (L1, L_inf, óptimos independientes)
    seed_pool: List[np.ndarray] = list(x0_candidates)
    if payoff.independent_solutions:
        for sol in payoff.independent_solutions:
            seed_pool.append(np.array([sol[name] for name in var_names], dtype=float) / column_scale)

    for cand in seed_pool:
        snapped = np.array(cand, dtype=float)
        for j in int_indices:
            snapped[j] = round(float(snapped[j]))
        if _is_feasible(snapped, A_ub, b_ub, A_eq, b_eq, bounds, integrality=integrality, tol=1e-7):
            val = objective(snapped)
            if val < best_int_val:
                best_int_val = val
                best_int_x = snapped

    # Resolver relajación raíz
    root_x, root_lb, root_certified = _solve_continuous_qp(scaled_alpha, beta, weights, A_ub, b_ub, A_eq, b_eq, bounds, x0_candidates)
    tree_exhausted = root_x is not None or root_certified
    if root_x is not None:
        start = time.monotonic()
        deadline = start + max_seconds
        if max_nodes > 0 and root_lb < best_int_val - 1e-7:
            # Las tangentes permiten saltar entre flujos enteros usando el
            # solver lineal entero, en vez de explorar todos los desgloses.
            best_int_x, best_int_val, certified = _refine_integer_qp(
                weights[:, None] * scaled_alpha, weights * beta,
                (A_ub, b_ub, A_eq, b_eq), bounds, integrality, [root_x, *seed_pool],
                (best_int_x, best_int_val), min(8, max_nodes), start + max_seconds / 2,
            )
            if certified:
                return report(best_int_x, True)

        # Cola de prioridad Best-First: (lower_bound, tie_breaker, node_bounds, node_x)
        heap: List[Tuple[float, int, List[Tuple[Optional[float], Optional[float]]], np.ndarray]] = []
        counter = 0
        heapq.heappush(heap, (root_lb, counter, bounds, root_x))

        nodes_explored = 0

        while heap and nodes_explored < max_nodes and time.monotonic() < deadline:
            lb_val, _, node_bounds, node_x = heapq.heappop(heap)
            if lb_val >= best_int_val - 1e-7:
                continue
            nodes_explored += 1

            # Buscar variable entera más fraccionaria
            branch_j = -1
            max_frac = 0.0
            for j in int_indices:
                frac = abs(node_x[j] - round(float(node_x[j])))
                if frac > 1e-7 and frac > max_frac:
                    max_frac = frac
                    branch_j = j

            if branch_j == -1:
                # Solución entera válida en este nodo
                snapped = np.array(node_x, dtype=float)
                for j in int_indices:
                    snapped[j] = round(float(snapped[j]))
                if _is_feasible(snapped, A_ub, b_ub, A_eq, b_eq, node_bounds, integrality=integrality, tol=1e-7):
                    val = objective(snapped)
                    if val < best_int_val:
                        best_int_val = val
                        best_int_x = snapped
                if lb_val < best_int_val - 1e-7:
                    tree_exhausted = False
                continue

            # Ramificar x_j <= floor y x_j >= ceil
            fl = float(np.floor(node_x[branch_j]))
            ce = float(np.ceil(node_x[branch_j]))
            orig_lb, orig_ub = node_bounds[branch_j]

            for child_lb, child_ub in [(orig_lb, fl if orig_ub is None else min(orig_ub, fl)), (ce if orig_lb is None else max(orig_lb, ce), orig_ub)]:
                if child_lb is not None and child_ub is not None and child_lb > child_ub + 1e-9:
                    continue
                c_bounds = list(node_bounds)
                c_bounds[branch_j] = (child_lb, child_ub)
                c_x, c_lb, c_certified = _solve_continuous_qp(scaled_alpha, beta, weights, A_ub, b_ub, A_eq, b_eq, c_bounds, [node_x])
                if c_x is None and not c_certified:
                    tree_exhausted = False
                if c_x is not None and c_lb < best_int_val - 1e-7:
                    counter += 1
                    heapq.heappush(heap, (c_lb, counter, c_bounds, c_x))

        # Verificar si quedaron nodos abiertos con cota inferior menor al incumbente
        while heap:
            rem_lb, _, _, _ = heapq.heappop(heap)
            if rem_lb < best_int_val - 1e-7:
                tree_exhausted = False
                break

    if best_int_x is None:
        return MetricSolution(
            metric="L2",
            variables={},
            objectives=[],
            achievement_pct=[],
            normalized_distances=[],
            aggregate_distance=0.0,
            status="error: no se encontró solución entera factible para L2",
        )

    return report(best_int_x, tree_exhausted)


def solve_all_metrics(
    model: ModelDefinition,
    payoff: PayoffMatrixData,
    weights: Optional[List[float]] = None
) -> Tuple[Dict[str, MetricSolution], List[float]]:
    """
    Calcula simultáneamente las tres soluciones compromiso (L1, L2, L_inf) para un vector de pesos dado.
    """
    n_objs = len(model.objectives)
    var_names = [v.name for v in model.variables]
    if weights is None or len(weights) != n_objs:
        w_array = np.ones(n_objs, dtype=float) / n_objs
    else:
        w_array = np.array(weights, dtype=float)
        w_array = np.maximum(0.0, w_array)
        s = np.sum(w_array)
        if s > 0:
            w_array = w_array / s
        else:
            w_array = np.ones(n_objs, dtype=float) / n_objs

    alpha, beta = compute_distance_coefficients(model, payoff)

    sol_l1 = solve_l1(model, payoff, alpha, beta, w_array)
    sol_linf = solve_linf(model, payoff, alpha, beta, w_array)

    incumbents: List[np.ndarray] = []
    if sol_l1.status == "optimal" and sol_l1.variables:
        incumbents.append(np.array([sol_l1.variables[n] for n in var_names], dtype=float))
    if sol_linf.status == "optimal" and sol_linf.variables:
        incumbents.append(np.array([sol_linf.variables[n] for n in var_names], dtype=float))

    # Punto inicial: punto medio factible entre L1 y L_inf (si ambos existen) para evitar estancamiento en vértices
    x0_l2: Optional[np.ndarray] = None
    if len(incumbents) == 2:
        x0_l2 = 0.5 * (incumbents[0] + incumbents[1])
    elif len(incumbents) == 1:
        x0_l2 = incumbents[0]

    sol_l2 = solve_l2(model, payoff, alpha, beta, w_array, x0=x0_l2, incumbents=incumbents)

    solutions = {
        "L1": sol_l1,
        "L2": sol_l2,
        "L_inf": sol_linf
    }
    return solutions, w_array.tolist()

"""
Generación y aproximación de la Frontera Eficiente de Pareto.
"""
import numpy as np
from typing import List

def is_dominated(point: List[float], other: List[float], senses: List[str]) -> bool:
    """
    Verifica si 'point' es dominado por 'other'.
    'other' domina a 'point' si es al menos tan bueno en todos los objetivos y estrictamente mejor en al menos uno.
    """
    better_or_equal = True
    strictly_better = False
    
    for val_p, val_o, sense in zip(point, other, senses):
        if sense == "maximize":
            if val_o < val_p - 1e-6:
                better_or_equal = False
                break
            if val_o > val_p + 1e-6:
                strictly_better = True
        else:  # minimize
            if val_o > val_p + 1e-6:
                better_or_equal = False
                break
            if val_o < val_p - 1e-6:
                strictly_better = True
                
    return better_or_equal and strictly_better

def compute_pareto_frontier(
    model: ModelDefinition,
    payoff: PayoffMatrixData,
    num_samples: int = 21
) -> List[ParetoPoint]:
    """
    Aproxima la frontera de Pareto barriendo sistemáticamente diferentes vectores de pesos
    y combinando múltiples métricas de compromiso (L1, L2 y L_inf).
    """
    n_objs = len(model.objectives)
    alpha, beta = compute_distance_coefficients(model, payoff)
    senses = [obj.sense for obj in model.objectives]
    var_names = [v.name for v in model.variables]

    weight_vectors = []

    if n_objs == 2:
        for w1 in np.linspace(0.0, 1.0, num_samples):
            weight_vectors.append(np.array([w1, 1.0 - w1]))
    elif n_objs == 3:
        steps = 10  # genera 66 combinaciones en el simplex 3D
        for i in range(steps + 1):
            w1 = i / steps
            for j in range(steps + 1 - i):
                w2 = j / steps
                w3 = (steps - i - j) / steps
                weight_vectors.append(np.array([w1, w2, w3]))
    else:
        # Para > 3 objetivos: vértices puros + centro equitativo + muestreo Dirichlet reproducible
        rng = np.random.default_rng(model.pareto_seed)
        weight_vectors.append(np.ones(n_objs) / n_objs)
        for k in range(n_objs):
            e_k = np.zeros(n_objs)
            e_k[k] = 1.0
            weight_vectors.append(e_k)
        for _ in range(max(num_samples, 36)):
            weight_vectors.append(rng.dirichlet(np.ones(n_objs)))

    raw_points: List[ParetoPoint] = []
    has_integer = any(v.var_type in ("integer", "binary") for v in model.variables)

    # Sembrar extremos individuales de la Matriz de Pagos para garantizar que los bordes A y B existan
    if payoff.matrix and payoff.independent_solutions:
        for k, (obj_vals, var_dict) in enumerate(zip(payoff.matrix, payoff.independent_solutions)):
            e_k = np.zeros(n_objs)
            e_k[k] = 1.0
            raw_points.append(
                ParetoPoint(
                    weights=e_k.tolist(),
                    objectives=list(obj_vals),
                    variables=dict(var_dict),
                    metric="L1"
                )
            )

    # Barrer pesos combinando métricas L1 (vértices), L2 (compromiso suave) y L_inf (barrido uniforme minimax)
    for w in weight_vectors:
        sol_l1 = solve_l1(model, payoff, alpha, beta, w)
        x0_seed = None
        if sol_l1.status == "optimal":
            x0_seed = np.array([sol_l1.variables[vn] for vn in var_names], dtype=float)
            raw_points.append(
                ParetoPoint(
                    weights=w.tolist(),
                    objectives=sol_l1.objectives,
                    variables=sol_l1.variables,
                    metric="L1"
                )
            )

        # El barrido minimax incluye pesos interiores cerca de los extremos de la cuadrícula.
        w_pos = np.maximum(w, 1e-3)
        w_pos = w_pos / np.sum(w_pos)
        sol_inf = solve_linf(model, payoff, alpha, beta, w_pos)
        if sol_inf.status == "optimal":
            raw_points.append(
                ParetoPoint(
                    weights=w_pos.tolist(),
                    objectives=sol_inf.objectives,
                    variables=sol_inf.variables,
                    metric="L_inf"
                )
            )

        if has_integer:
            sol_l2 = solve_l2(
                model,
                payoff,
                alpha,
                beta,
                w,
                x0=x0_seed,
                max_nodes=80,
                max_seconds=0.05,
            )
        else:
            sol_l2 = solve_l2(model, payoff, alpha, beta, w, x0=x0_seed)
        if sol_l2.status in ("optimal", "feasible_suboptimal"):
            raw_points.append(
                ParetoPoint(
                    weights=w.tolist(),
                    objectives=sol_l2.objectives,
                    variables=sol_l2.variables,
                    metric="L2",
                    status=sol_l2.status,
                )
            )

    # Filtrar soluciones dominadas y deduplicar manteniendo diversidad entre métricas L1 / L2 / L_inf
    non_dominated: List[ParetoPoint] = []
    seen_coords: List[np.ndarray] = []
    ideal = np.array(payoff.ideal_point)
    scales = np.array(payoff.normalization_ranges or [1.0] * n_objs)

    # Primero quedarnos con todos los puntos no dominados
    efficient_candidates: List[ParetoPoint] = []
    normalized = [(np.array(p.objectives) - ideal) / scales for p in raw_points]
    for p, coords in zip(raw_points, normalized):
        if not any(is_dominated(coords, other, senses) for other in normalized):
            efficient_candidates.append(p)

    # Priorizar que queden representadas todas las métricas (L1 en extremos, L2 y L_inf en interiores)
    for p in sorted(efficient_candidates, key=lambda p: p.status != "optimal"):
        obj_vals = (np.array(p.objectives) - ideal) / scales
        if any(np.max(np.abs(obj_vals - s)) <= 1e-5 for s in seen_coords):
            continue
        non_dominated.append(p)
        seen_coords.append(obj_vals)

    # Ordenar en 2D por el primer objetivo para trazar la curva continua sin zigzags
    if n_objs == 2:
        non_dominated.sort(key=lambda p: p.objectives[0])

    return non_dominated

def resolver_opticompromise(model, weights=None, compute_pareto=True):
    definition = ModelDefinition.model_validate(model)
    payoff = compute_payoff_matrix(definition)
    if payoff.status != "optimal":
        raise ValueError(payoff.status)
    solutions, applied_weights = solve_all_metrics(
        definition, payoff, definition.weights if weights is None else weights,
    )
    points = compute_pareto_frontier(definition, payoff) if compute_pareto else []
    return payoff, solutions, applied_weights, points

In [ ]:
# 3. Casos de Validación Oficiales del Curso y Modelos Bonus
MODELOS_EJEMPLO = {
  "1. Celdas Fotovoltaicas (Clase 15 · Curso)": {
    "name": "Planificación de Celdas Fotovoltaicas (Clase 15)",
    "description": "Problema biobjetivo del curso: maximizar utilidad económica vs minimizar generación de contaminantes.",
    "variables": [
      {
        "name": "x1 (Celdas Fotov.)",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 300.0
      },
      {
        "name": "x2 (Celdas GP)",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 200.0
      }
    ],
    "objectives": [
      {
        "name": "Utilidad Total (USD)",
        "sense": "maximize",
        "coefficients": [
          1000.0,
          3000.0
        ]
      },
      {
        "name": "Emisión Contaminantes (kg)",
        "sense": "minimize",
        "coefficients": [
          1.0,
          2.0
        ]
      }
    ],
    "constraints": [
      {
        "name": "Capacidad Máxima de Ensamble",
        "coefficients": [
          1.0,
          1.0
        ],
        "sense": "<=",
        "rhs": 400.0
      },
      {
        "name": "Demanda Mínima de Ingreso",
        "coefficients": [
          1000.0,
          3000.0
        ],
        "sense": ">=",
        "rhs": 300000.0
      }
    ],
    "weights": [
      0.5,
      0.5
    ]
  },
  "2. Problema de Transporte (Taller 3 · 12 Variables)": {
    "name": "Problema de Transporte Multiobjetivo (Taller 3)",
    "description": "Distribución desde 3 orígenes (O1, O2, O3) hacia 4 destinos (D1, D2, D3, D4), minimizando costo total de flete y minimizando riesgo en rutas.",
    "variables": [
      {
        "name": "x_1_1 (O1 → D1)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_1_2 (O1 → D2)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_1_3 (O1 → D3)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_1_4 (O1 → D4)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_2_1 (O2 → D1)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_2_2 (O2 → D2)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_2_3 (O2 → D3)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_2_4 (O2 → D4)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_3_1 (O3 → D1)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_3_2 (O3 → D2)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_3_3 (O3 → D3)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_3_4 (O3 → D4)",
        "var_type": "continuous",
        "lower_bound": 0.0
      }
    ],
    "objectives": [
      {
        "name": "Costo de Transporte (COP)",
        "sense": "minimize",
        "coefficients": [
          4.0,
          3.0,
          3.0,
          4.0,
          5.0,
          2.0,
          1.0,
          5.0,
          7.0,
          10.0,
          8.0,
          9.0
        ]
      },
      {
        "name": "Riesgo Global de Transporte",
        "sense": "minimize",
        "coefficients": [
          6.0,
          0.0,
          1.0,
          5.0,
          2.0,
          4.0,
          0.0,
          2.0,
          3.0,
          1.0,
          2.0,
          1.0
        ]
      }
    ],
    "constraints": [
      {
        "name": "Oferta Origen O1",
        "coefficients": [
          1.0,
          1.0,
          1.0,
          1.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0
        ],
        "sense": "<=",
        "rhs": 800.0
      },
      {
        "name": "Oferta Origen O2",
        "coefficients": [
          0.0,
          0.0,
          0.0,
          0.0,
          1.0,
          1.0,
          1.0,
          1.0,
          0.0,
          0.0,
          0.0,
          0.0
        ],
        "sense": "<=",
        "rhs": 1200.0
      },
      {
        "name": "Oferta Origen O3",
        "coefficients": [
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          1.0,
          1.0,
          1.0,
          1.0
        ],
        "sense": "<=",
        "rhs": 1500.0
      },
      {
        "name": "Demanda Destino D1",
        "coefficients": [
          1.0,
          0.0,
          0.0,
          0.0,
          1.0,
          0.0,
          0.0,
          0.0,
          1.0,
          0.0,
          0.0,
          0.0
        ],
        "sense": ">=",
        "rhs": 500.0
      },
      {
        "name": "Demanda Destino D2",
        "coefficients": [
          0.0,
          1.0,
          0.0,
          0.0,
          0.0,
          1.0,
          0.0,
          0.0,
          0.0,
          1.0,
          0.0,
          0.0
        ],
        "sense": ">=",
        "rhs": 700.0
      },
      {
        "name": "Demanda Destino D3",
        "coefficients": [
          0.0,
          0.0,
          1.0,
          0.0,
          0.0,
          0.0,
          1.0,
          0.0,
          0.0,
          0.0,
          1.0,
          0.0
        ],
        "sense": ">=",
        "rhs": 900.0
      },
      {
        "name": "Demanda Destino D4",
        "coefficients": [
          0.0,
          0.0,
          0.0,
          1.0,
          0.0,
          0.0,
          0.0,
          1.0,
          0.0,
          0.0,
          0.0,
          1.0
        ],
        "sense": ">=",
        "rhs": 1400.0
      }
    ],
    "weights": [
      0.5,
      0.5
    ]
  },
  "3. Sostenibilidad Triobjetivo 3D (Bonus 3Z)": {
    "name": "Optimización Sostenible Triobjetivo 3D (Bonus)",
    "description": "Problema con 3 dimensiones de impacto: maximizar retorno económico, minimizar impacto ambiental y maximizar beneficio social.",
    "variables": [
      {
        "name": "x1 (Línea Rentable)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x2 (Línea Social)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x3 (Línea Intensiva)",
        "var_type": "continuous",
        "lower_bound": 0.0
      }
    ],
    "objectives": [
      {
        "name": "Retorno Económico (kUSD)",
        "sense": "maximize",
        "coefficients": [
          3.0,
          1.0,
          2.0
        ]
      },
      {
        "name": "Impacto Ambiental (CO2 eq)",
        "sense": "minimize",
        "coefficients": [
          2.0,
          1.0,
          4.0
        ]
      },
      {
        "name": "Beneficio Social (Puntos)",
        "sense": "maximize",
        "coefficients": [
          1.0,
          3.0,
          1.0
        ]
      }
    ],
    "constraints": [
      {
        "name": "Presupuesto de Capital",
        "coefficients": [
          1.0,
          1.0,
          1.0
        ],
        "sense": "<=",
        "rhs": 100.0
      },
      {
        "name": "Capacidad Técnica Operativa A",
        "coefficients": [
          2.0,
          1.0,
          0.0
        ],
        "sense": "<=",
        "rhs": 120.0
      },
      {
        "name": "Capacidad Técnica Operativa B",
        "coefficients": [
          0.0,
          1.0,
          2.0
        ],
        "sense": "<=",
        "rhs": 140.0
      }
    ],
    "weights": [
      0.3333,
      0.3333,
      0.3334
    ]
  },
  "4. Producción Mixta MIP (Bonus Entera/Binaria)": {
    "name": "Planificación Entera y Binaria (MIP Bonus)",
    "description": "Optimización de rendimiento vs costo operativo con variables continuas (lotes), enteras (máquinas) y binaria (apertura de planta).",
    "variables": [
      {
        "name": "x1",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 50.0
      },
      {
        "name": "x2",
        "var_type": "integer",
        "lower_bound": 0.0,
        "upper_bound": 20.0
      },
      {
        "name": "y1",
        "var_type": "binary",
        "lower_bound": 0.0,
        "upper_bound": 1.0
      }
    ],
    "objectives": [
      {
        "name": "Rendimiento (USD)",
        "sense": "maximize",
        "coefficients": [
          40.0,
          70.0,
          150.0
        ]
      },
      {
        "name": "Costo Operativo (USD)",
        "sense": "minimize",
        "coefficients": [
          10.0,
          25.0,
          80.0
        ]
      }
    ],
    "constraints": [
      {
        "name": "Capacidad Planta",
        "coefficients": [
          1.0,
          2.0,
          -30.0
        ],
        "sense": "<=",
        "rhs": 0.0
      },
      {
        "name": "Presupuesto Global",
        "coefficients": [
          10.0,
          25.0,
          80.0
        ],
        "sense": "<=",
        "rhs": 500.0
      },
      {
        "name": "Demanda Mínima",
        "coefficients": [
          1.0,
          1.0,
          0.0
        ],
        "sense": ">=",
        "rhs": 15.0
      }
    ],
    "weights": [
      0.6,
      0.4
    ]
  },
  "5. Flota de Vehículos Eléctricos (6 Autos, 4 Objetivos)": {
    "name": "Adquisición de Flota de Vehículos Eléctricos (Caso Taller IAD)",
    "description": "Evaluación multicriterio para adquisición de flota entre 6 alternativas de vehículos 100% eléctricos: BYD Seal, Tesla Model 3, Hyundai Ioniq 5, Kia EV6, Volvo EX30 y Chevrolet Equinox EV.",
    "variables": [
      {
        "name": "x1 (BYD Seal)",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 50.0
      },
      {
        "name": "x2 (Tesla Model 3)",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 50.0
      },
      {
        "name": "x3 (Hyundai Ioniq 5)",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 50.0
      },
      {
        "name": "x4 (Kia EV6)",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 50.0
      },
      {
        "name": "x5 (Volvo EX30)",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 50.0
      },
      {
        "name": "x6 (Chevrolet Equinox EV)",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 50.0
      }
    ],
    "objectives": [
      {
        "name": "Costo Total Flota (USD)",
        "sense": "minimize",
        "coefficients": [
          44000.0,
          46500.0,
          49000.0,
          48000.0,
          42000.0,
          43500.0
        ]
      },
      {
        "name": "Autonomía Total (km)",
        "sense": "maximize",
        "coefficients": [
          570.0,
          630.0,
          520.0,
          540.0,
          480.0,
          510.0
        ]
      },
      {
        "name": "Tiempo acumulado de carga (min)",
        "sense": "minimize",
        "coefficients": [
          32.0,
          27.0,
          18.0,
          18.0,
          26.0,
          30.0
        ]
      },
      {
        "name": "Índice Seguridad Flota (pts)",
        "sense": "maximize",
        "coefficients": [
          9.0,
          9.5,
          9.6,
          9.7,
          10.0,
          9.3
        ]
      }
    ],
    "constraints": [
      {
        "name": "Tamaño Requerido de Flota",
        "coefficients": [
          1.0,
          1.0,
          1.0,
          1.0,
          1.0,
          1.0
        ],
        "sense": "==",
        "rhs": 50.0
      },
      {
        "name": "Capacidad Mínima Maletero (L)",
        "coefficients": [
          400.0,
          594.0,
          527.0,
          520.0,
          318.0,
          748.0
        ],
        "sense": ">=",
        "rhs": 24000.0
      },
      {
        "name": "Presupuesto Máximo Inversión (USD)",
        "coefficients": [
          44000.0,
          46500.0,
          49000.0,
          48000.0,
          42000.0,
          43500.0
        ],
        "sense": "<=",
        "rhs": 2350000.0
      }
    ],
    "weights": [
      0.3,
      0.25,
      0.25,
      0.2
    ]
  }
}
print(f'✓ {len(MODELOS_EJEMPLO)} modelos precargados listos.')


In [ ]:
# 4. Panel Interactivo en Colab (Deslizadores dinámicos para todos los objetivos + Gráficas)
dropdown_modelo = widgets.Dropdown(
    options=list(MODELOS_EJEMPLO.keys()),
    description="Modelo:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="560px")
)
weights_box = widgets.VBox()
output_dashboard = widgets.Output()
sliders_refs = []

def render_dashboard(*args):
    m_data = MODELOS_EJEMPLO[dropdown_modelo.value]
    raw_w = [s.value for s in sliders_refs]
    with output_dashboard:
        clear_output(wait=True)
        try:
            payoff, solutions, w_norm, points = resolver_opticompromise(m_data, weights=raw_w)
        except ValueError as error:
            print(error)
            return
        pareto = [p.objectives for p in points]
        obj_names = [o["name"] for o in m_data["objectives"]]
        var_names = [v["name"] for v in m_data["variables"]]

        display(HTML(f"<h3>📊 1. Matriz de Pagos y Puntos de Referencia ({m_data['name']})</h3>"))
        df_payoff = pd.DataFrame(payoff.matrix, index=[f"Óptimo {n}" for n in obj_names], columns=obj_names)
        df_payoff.loc["Punto Ideal (Z*)"] = payoff.ideal_point
        df_payoff.loc["Nadir aproximado"] = payoff.anti_ideal_point
        display(df_payoff.round(2))
        if payoff.degenerate_objectives:
            scale_labels = {"feasible_range": "rango factible", "constant": "brecha cero", "unit": "una unidad"}
            for i in payoff.degenerate_objectives:
                print(f"{obj_names[i]} · rango de pagos nulo. Escala: {scale_labels[payoff.normalization_sources[i]]}.")
        if any(p.status != "optimal" for p in points):
            print("La muestra de Pareto incluye puntos factibles sin certificado de optimalidad para su métrica.")

        labels = {"L1": "L₁", "L2": "L₂", "L_inf": "L∞"}
        colors = {"L1": "#2563EB", "L2": "#334155", "L_inf": "#E11D48"}
        markers = {"L1": "s", "L2": "o", "L_inf": "D"}
        rows = []
        available = {key: sol for key, sol in solutions.items() if sol.objectives}
        for key, sol in solutions.items():
            state = {
                "optimal": "Óptimo dentro de tolerancia",
                "feasible_suboptimal": "Factible sin certificado",
            }.get(sol.status, sol.status)
            item = {"Métrica": labels[key], "Estado": state}
            if sol.objectives:
                item["Distancia D_p"] = round(sol.aggregate_distance, 4)
                item["Variables x"] = ", ".join(f"{name}={sol.variables[name]:.2f}" for name in var_names)
                for idx, oname in enumerate(obj_names):
                    item[oname] = round(sol.objectives[idx], 2)
                    item[f"% Logro Z'{idx+1}"] = f"{sol.achievement_pct[idx]:.2f}%"
            rows.append(item)
        display(HTML(f"<h3>🎯 2. Soluciones Compromiso (Pesos normalizados: {[round(float(v), 3) for v in w_norm]})</h3>"))
        display(pd.DataFrame(rows))

        n_objs = len(obj_names)
        is_mip = any(v.get("var_type", "continuous") != "continuous" for v in m_data["variables"])

        if n_objs == 2:
            plt.figure(figsize=(8.5, 4.8))
            px = [p[0] for p in pareto]; py = [p[1] for p in pareto]
            if px:
                if is_mip:
                    plt.scatter(px, py, color="#0F766E", s=40, alpha=0.75, label="Muestra Pareto (MIP)")
                else:
                    plt.plot(px, py, color="#E11D48", lw=2.2, alpha=0.85, label="Unión de muestras Pareto", marker=".")
            plt.scatter(payoff.ideal_point[0], payoff.ideal_point[1], color="#D97706", s=220, marker="*", label="Ideal (Z*)", zorder=6)
            plt.scatter(payoff.anti_ideal_point[0], payoff.anti_ideal_point[1], color="#BE123C", s=110, marker="X", label="Nadir aproximado", zorder=6)
            for key, sol in available.items():
                plt.scatter(*sol.objectives, color=colors[key], s=120, marker=markers[key], label=labels[key], zorder=7)
            plt.title(f"Espacio de Objetivos: {m_data['name']}", fontweight="bold")
            plt.xlabel(obj_names[0]); plt.ylabel(obj_names[1])
            plt.grid(True, linestyle="--", alpha=0.5); plt.legend(loc="best"); plt.tight_layout(); plt.show()
        elif n_objs == 3:
            fig = plt.figure(figsize=(8.5, 5.5))
            ax = fig.add_subplot(111, projection="3d")
            if pareto:
                ax.scatter([p[0] for p in pareto], [p[1] for p in pareto], [p[2] for p in pareto], color="#0F766E", s=24, alpha=0.5, label="Muestra Pareto 3D")
            ax.scatter([payoff.ideal_point[0]], [payoff.ideal_point[1]], [payoff.ideal_point[2]], color="#D97706", s=180, marker="*", label="Ideal (Z*)")
            ax.scatter([payoff.anti_ideal_point[0]], [payoff.anti_ideal_point[1]], [payoff.anti_ideal_point[2]], color="#BE123C", s=100, marker="X", label="Nadir aproximado")
            for key, sol in available.items():
                ax.scatter(*sol.objectives, color=colors[key], s=110, marker=markers[key], label=labels[key])
            ax.set_xlabel(obj_names[0]); ax.set_ylabel(obj_names[1]); ax.set_zlabel(obj_names[2])
            ax.set_title(f"Espacio 3D: {m_data['name']}", fontweight="bold")
            ax.legend(loc="best"); plt.tight_layout(); plt.show()
        else:
            N = n_objs
            angles = [n / float(N) * 2 * np.pi for n in range(N)] + [0.0]
            fig, ax = plt.subplots(figsize=(6.5, 6.5), subplot_kw=dict(polar=True))
            plt.xticks(angles[:-1], obj_names, size=9, fontweight="bold")
            ax.set_ylim(min([0] + [v for sol in available.values() for v in sol.achievement_pct]), 108)
            ax.plot(angles, [100]*(N+1), lw=1.5, ls="--", color="#D97706", label="Ideal (100%)")
            for key, sol in available.items():
                vals = sol.achievement_pct + sol.achievement_pct[:1]
                ax.plot(angles, vals, lw=2, color=colors[key], label=labels[key])
                ax.fill(angles, vals, colors[key], alpha=0.1)
            plt.title(f"Perfil Multicriterio (% Logro): {m_data['name']}", fontweight="bold", y=1.08)
            plt.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1)); plt.tight_layout(); plt.show()

def rebuild_sliders(*args):
    global sliders_refs
    m_data = MODELOS_EJEMPLO[dropdown_modelo.value]
    n_objs = len(m_data["objectives"])
    def_w = m_data.get("weights") or [1.0 / n_objs] * n_objs
    sliders_refs = []
    for i, o in enumerate(m_data["objectives"]):
        sl = widgets.FloatSlider(
            value=float(def_w[i]), min=0.0, max=1.0, step=0.05,
            description=f"w{i+1} ({o['name'][:22]}):",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="520px")
        )
        sl.observe(render_dashboard, names="value")
        sliders_refs.append(sl)
    weights_box.children = tuple(sliders_refs)
    render_dashboard()

dropdown_modelo.observe(rebuild_sliders, names="value")
display(HTML("<h2>🚀 OptiCompromise — Cuaderno Interactivo UNAL</h2>"))
display(dropdown_modelo, weights_box, output_dashboard)
rebuild_sliders()
